# Тестирование модуля Algo

Этот ноутбук демонстрирует базовую работу алгоритмического модуля: токенизацию, сравнение слов и посимвольный diff.

In [1]:
import sys
import os

# Добавляем корень проекта в путь, чтобы видеть пакет 'algo'
sys.path.append(os.path.abspath(os.path.join('..', '..')))

from algo import tokenize_paragraphs, compare_tokens, char_diff, Weights
from algo.structures import WordOpType, CharOpType

## 1. Токенизация
Проверим, как текст разбивается на токены. Знаки препинания должны быть отдельными токенами.

In [4]:
paragraphs = [
    "1 2 3",
    "4 5 6"
]

res = tokenize_paragraphs(paragraphs)
print("Tokens:", res.tokens)
print("Paragraph indices:", res.token_paragraph_index)

Tokens: ['1', '2', '3', '4', '5', '6']
Paragraph indices: [0, 0, 0, 1, 1, 1]


## 2. Сравнение слов (Word Diff)
Сравним два простых предложения.

In [5]:
text1 = ["1", "0", "3", "4"]
text2 = ["1", "2", "3", "4", "6"]

weights = Weights(replace=1.0, delete=1.0, insert=1.0)

ops = compare_tokens(text1, text2, weights)

for op in ops:
    if op.op_type == WordOpType.equal:
        print(f"  {op.base_token}")
    elif op.op_type == WordOpType.insert:
        print(f"+ {op.other_token}")
    elif op.op_type == WordOpType.delete:
        print(f"- {op.base_token}")
    elif op.op_type == WordOpType.replace:
        print(f"~ {op.base_token} -> {op.other_token}")

  1
~ 0 -> 2
  3
  4
+ 6


## 3. Посимвольное сравнение (Char Diff)
Проверим, как работает `char_diff` на примере замены слова.

In [6]:
word_a = "123"
word_b = "1034"

c_ops = char_diff(word_a, word_b)

print(f"Diff '{word_a}' -> '{word_b}':")
for op in c_ops:
    if op.op_type == CharOpType.change:
        print(f"Change: {op.from_str} -> {op.to_str}")
    elif op.op_type == CharOpType.insert:
        print(f"Insert: {op.to_str}")
    elif op.op_type == CharOpType.delete:
        print(f"Delete: {op.from_str}")
    else:
        print(op)

Diff '123' -> '1034':
Change: 2 -> 0
Insert: 4


In [7]:
# Пример с ё
word_a = "елка"
word_b = "ёлка"

c_ops = char_diff(word_a, word_b)
for op in c_ops:
    print(f"{op.op_type}: '{op.from_str}' -> '{op.to_str}'")

change: 'е' -> 'ё'


In [8]:
import re
from typing import List

from algo.structures import TokenizationResult

TOKEN_REGEX = re.compile(r'\S+')

def tokenize_paragraphs2(paragraphs: List[str]) -> TokenizationResult:
    all_tokens = []
    token_indices = []

    for idx, para in enumerate(paragraphs):
        # We assume 0-based index here, UI adds 1
        matches = TOKEN_REGEX.findall(para)
        for token in matches:
            all_tokens.append(token)
            token_indices.append(idx)

    return TokenizationResult(
        tokens=all_tokens,
        token_paragraph_index=token_indices
    )

In [10]:
## 4. Загрузка текстов и шрифта из файлов
# Пример того, как загружаются файлы для последующей обработки

# Basic tokenizer: words (letters/numbers/underscore) or single punctuation characters
# Adjust regex as needed for "word" definition.
# Spec: "punctuation - separate tokens", "spaces are not tokens".


import docx

# Пути к файлам
data_dir = os.path.abspath(os.path.join('..', '..', 'data'))
text1_path = os.path.join(data_dir, 'M10_1.docx')
text2_path = os.path.join(data_dir, 'U3_1.docx')
# text1_path = os.path.join(data_dir, 'text1.txt')
# text2_path = os.path.join(data_dir, 'text2.txt')
# text1_path = os.path.join(data_dir, 'text6.docx')
# text2_path = os.path.join(data_dir, 'text7.docx')

font_dir = os.path.abspath(os.path.join('..', '..', 'uploads', 'fonts', '1')) # Пример для юзера 1
# Ищем любой шрифт
font_path = None
if os.path.exists(font_dir):
    files = [f for f in os.listdir(font_dir) if f.endswith(('.ttf', '.otf', '.woff'))]
    if files:
        font_path = os.path.join(font_dir, files[0])

# Чтение
print(f"Reading Text 1: {text1_path}")
if os.path.exists(text1_path):
    if text1_path.endswith('.docx'):
        doc = docx.Document(text1_path)
        # Получаем весь текст из документа
        t1 = '\n'.join([paragraph.text for paragraph in doc.paragraphs])
    else:
        # Для обычных текстовых файлов
        with open(text1_path, 'r', encoding='utf-8') as f:
            t1 = f.read()
else:
    t1 = "Пример текста 1"
    print("File not found, using dummy text")

print(f"Reading Text 2: {text2_path}")

if os.path.exists(text2_path):
    if text2_path.endswith('.docx'):
        doc = docx.Document(text2_path)
        # Получаем весь текст из документа
        t2 = '\n'.join([paragraph.text for paragraph in doc.paragraphs])
    else:
        # Для обычных текстовых файлов
        with open(text2_path, 'r', encoding='utf-8') as f:
            t2 = f.read()
else:
    t2 = "Пример текста 2"
    print("File not found, using dummy text")

print(f"Font file: {font_path if font_path else 'Not found'}")

# Запуск сравнения
print("\nRunning comparison...")
p1 = [l.strip() for l in t1.split('\n') if l.strip()]
p2 = [l.strip() for l in t2.split('\n') if l.strip()]

print(p2)

tok1 = tokenize_paragraphs2(p1)
tok2 = tokenize_paragraphs2(p2)
ops = compare_tokens(tok1.tokens, tok2.tokens, Weights())

print()
for item in tok2.tokens:
    print(item)
print()

print(f"Found {len(ops)} word operations.")

for item in ops:
    print(item.base_token, item.other_token)

if font_path:
    print(f"Ready to render with font: {os.path.basename(font_path)}")

Reading Text 1: /Users/ehlamkin/Desktop/nir_2026_v2/data/M10_1.docx
Reading Text 2: /Users/ehlamkin/Desktop/nir_2026_v2/data/U3_1.docx
Font file: /Users/ehlamkin/Desktop/nir_2026_v2/uploads/fonts/1/a7219f26-1662-4da9-bdb6-4ba041e3d1b3.ttf

Running comparison...
['возвесели с высоты ѹтвръженїемъ сл҃вы']

возвесели
с
высоты
ѹтвръженїемъ
сл҃вы

Found 9 word operations.
възвеселе возвесели
None с
высоты высоты
ѹтвержⷣенїа ѹтвръженїемъ
оцѣщеⷩа None
зрⷦ None
нб҃сї None
вїдѣнїеⷨ None
сл҃вы сл҃вы
Ready to render with font: a7219f26-1662-4da9-bdb6-4ba041e3d1b3.ttf


In [7]:
from algo.comparison import compare_tokens_levenshtein_nested

# Предполагается, что функции compare_tokens_levenshtein_nested импортирована выше

M10_1 = "възвеселе высоты ѹтвержⷣенїа оцѣщеⷩа зрⷦ нб҃сї вїдѣнїеⷨ сл҃вы"
U3_1 = "возвесели с высоты ѹтвръженїемъ сл҃вы"

# Если нужно разбить на токены аналогично общему пайплайну:
M10_1_paragraphs = [l.strip() for l in M10_1.split('\n') if l.strip()]
U3_1_paragraphs = [l.strip() for l in U3_1.split('\n') if l.strip()]

# Можно использовать tokenize_paragraphs2, если определено, иначе .split():
try:
    tokens_M10_1 = tokenize_paragraphs2(M10_1_paragraphs).tokens
    tokens_U3_1 = tokenize_paragraphs2(U3_1_paragraphs).tokens
except Exception:
    tokens_M10_1 = M10_1.split()
    tokens_U3_1 = U3_1.split()

weights = Weights(3,1,1)  # если нужны другие веса, можно поменять

print(tokens_M10_1)
print(tokens_U3_1)

lev_nested_ops = compare_tokens_levenshtein_nested(tokens_M10_1, tokens_U3_1, weights)
print("Слово-операции для M10_1 <-> U3_1 (levenshtein_nested):")
for op in lev_nested_ops:
    print(op)


['възвеселе', 'высоты', 'ѹтвержⷣенїа', 'оцѣщеⷩ\ue003а', 'зрⷦ\ue003', 'нб҃сї', 'вїдѣнїеⷨ', 'сл҃вы']
['возвесели', 'с', 'высоты', 'ѹтвръженїемъ', 'сл҃вы']
char dist:  възвеселе возвесели 0.6666666666666666 2.0 2.0
char dist:  възвеселе с 3.6666666666666665 3.0 1.6666666666666665
char dist:  възвеселе высоты 4.333333333333334 4.0 2.6666666666666665
char dist:  възвеселе ѹтвръженїемъ 5.25 5.0 3.6666666666666665
char dist:  възвеселе сл҃вы 6.666666666666666 6.0 4.666666666666666
char dist:  высоты возвесели 3.3333333333333335 1.6666666666666665 3.0
char dist:  высоты с 3.1666666666666665 2.6666666666666665 2.6666666666666665
char dist:  высоты высоты 1.6666666666666665 3.6666666666666665 3.6666666666666665
char dist:  высоты ѹтвръженїемъ 5.416666666666666 4.666666666666666 2.6666666666666665
char dist:  высоты сл҃вы 6.166666666666666 5.666666666666666 3.6666666666666665
char dist:  ѹтвержⷣенїа возвесели 4.454545454545455 2.6666666666666665 4.0
char dist:  ѹтвержⷣенїа с 4.666666666666666 3.6

In [11]:
def levenshtein_distance_positions(list1, list2, del_cost=1, ins_cost=1, sub_cost=1):

    if isinstance(list1, str):
        list1 = list1.split()
        list2 = list2.split()

    len1, len2 = len(list1), len(list2)
    dp = [[0] * (len2 + 1) for _ in range(len1 + 1)]

    # Инициализация первой строки и столбца
    for i in range(1, len1 + 1):
        dp[i][0] = i * del_cost
    for j in range(1, len2 + 1):
        dp[0][j] = j * ins_cost

    # Заполнение таблицы
    for i in range(1, len1 + 1):
        for j in range(1, len2 + 1):
            if list1[i - 1] == list2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]  # Без стоимости, если элементы совпадают
            else:
                dp[i][j] = min(
                    dp[i - 1][j] + del_cost,      # Удаление
                    dp[i][j - 1] + ins_cost,      # Вставка
                    dp[i - 1][j - 1] + sub_cost   # Замена
                )

    # Обратное восстановление пути
    i, j = len1, len2
    res_dict = {
        "distance": dp[len1][len2],
        "delete": [],
        "insert": [],
        "change": []
    }

    while i > 0 or j > 0:
        if i > 0 and j > 0 and list1[i - 1] == list2[j - 1]:
            i -= 1
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + del_cost:
            res_dict["delete"].append((i - 1,))
            i -= 1
        elif j > 0 and dp[i][j] == dp[i][j - 1] + ins_cost:
            res_dict["insert"].append((j - 1,))
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + sub_cost:
            res_dict["change"].append((i - 1, j - 1))
            i -= 1
            j -= 1

    return res_dict

In [14]:
levenshtein_distance_positions(["1", "2", "3"], ["1", "0", "3", "4"])

{'distance': 2, 'delete': [], 'insert': [(3,)], 'change': [(1, 1)]}